In [5]:
import pandas as pd
df = pd.read_csv("osv_processed_features.csv", low_memory=False)
print(df["country"].dropna().unique()[:50])

['AD' 'AE' 'AG' 'AL' 'AM' 'AO' 'AR' 'AT' 'AU' 'AX' 'AZ' 'BA' 'BB' 'BD'
 'BE' 'BF' 'BG' 'BH' 'BI' 'BJ' 'BN' 'BO' 'BR' 'BS' 'BW' 'BY' 'BZ' 'CA'
 'CD' 'CF' 'CH' 'CL' 'CM' 'CN' 'CO' 'CR' 'CU' 'CY' 'CZ' 'DE' 'DK' 'DO'
 'DZ' 'EC' 'EE' 'EG' 'ES' 'ET' 'FI' 'FM']


In [2]:
# build_continent_from_iso.py
import pandas as pd
import pycountry_convert as pc

def code_to_continent(country_code):
    """Convert 2-letter ISO country code to continent name."""
    try:
        continent_code = pc.country_alpha2_to_continent_code(country_code)
        return {
            "AF": "Africa",
            "AS": "Asia",
            "EU": "Europe",
            "NA": "North America",
            "SA": "South America",
            "OC": "Oceania"
        }[continent_code]
    except Exception:
        return None

# --- Load your data ---
df = pd.read_csv("osv_processed_features.csv", low_memory=False)

# --- Add continent column ---
df["continent"] = df["country"].apply(code_to_continent)

# --- Check distribution ---
print(df["continent"].value_counts(dropna=False))
print("\n✅ Preview:")
print(df[["country","continent"]].head(10))

# --- Save updated file ---
df.to_csv("osv_with_continent.csv", index=False)
print("\n✅ Saved osv_with_continent.csv with new 'continent' column.")

continent
Europe           4155
North America    2425
Asia             1462
South America     626
Africa            406
Oceania           395
None               11
Name: count, dtype: int64

✅ Preview:
  country continent
0      AD    Europe
1      AE      Asia
2      AE      Asia
3      AE      Asia
4      AE      Asia
5      AE      Asia
6      AE      Asia
7      AE      Asia
8      AE      Asia
9      AE      Asia

✅ Saved osv_with_continent.csv with new 'continent' column.


In [3]:
# RouterDataset.py
import os
import torch
from torch.utils.data import Dataset
from PIL import Image

class ContinentDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

        # Only keep samples with valid continent labels
        self.df = self.df.dropna(subset=["continent"])

        # Encode text continent → integer class ID
        self.continent_to_idx = {c: i for i, c in enumerate(sorted(self.df["continent"].unique()))}
        self.idx_to_continent = {v: k for k, v in self.continent_to_idx.items()}
        self.df["continent_id"] = self.df["continent"].map(self.continent_to_idx)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, f"{row['id']}.jpg")
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(row["continent_id"], dtype=torch.long)
        return image, target

In [4]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from torchvision import transforms

class ContinentDataModule(pl.LightningDataModule):
    def __init__(self, df, image_dir, batch_size=16):
        super().__init__()
        self.df = df
        self.image_dir = image_dir
        self.batch_size = batch_size
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def setup(self, stage=None):
        from sklearn.model_selection import train_test_split
        train_df, val_df = train_test_split(self.df, test_size=0.2, random_state=42, stratify=self.df["continent"])
        self.train_dataset = ContinentDataset(train_df, self.image_dir, self.transform)
        self.val_dataset = ContinentDataset(val_df, self.image_dir, self.transform)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=4)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)

In [5]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

class RouterNet(pl.LightningModule):
    def __init__(self, num_classes=6, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = models.resnet18(weights="IMAGENET1K_V1")
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log_dict({"train_loss": loss, "train_acc": acc})
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log_dict({"val_loss": loss, "val_acc": acc}, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.hparams.lr)

In [6]:
import pandas as pd
from pytorch_lightning.callbacks import ModelCheckpoint
df = pd.read_csv("osv_with_continent.csv")
df = df.dropna(subset=["continent"]).reset_index(drop=True)
data_module = ContinentDataModule(df, image_dir="osv-5m_subset/train", batch_size=16)
model = RouterNet(num_classes=df["continent"].nunique(), lr=1e-4)

checkpoint_callback = ModelCheckpoint(
    dirpath="outputs/router_checkpoints",  # Folder to store checkpoints
    filename="RouterNet-{epoch:02d}-{val_acc:.3f}",  # Name includes accuracy
    monitor="val_acc",      # Track validation accuracy
    mode="max",             # Keep the one with the highest accuracy
    save_top_k=1,           # Save only the best model
    save_weights_only=True  # Skip optimizer state; smaller file
)

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    log_every_n_steps=10,
    callbacks=[checkpoint_callback]
)

trainer.fit(model, data_module)
trainer.validate(model, data_module)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/work/cssema416/202610/28/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
/work/cssema416/202610/28/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /work/cssema416/202610/28/outputs/router_checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]

  | Name      | Type             | Params | Mode 
-------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         val_acc            0.6927138566970825
        val_loss            1.3518352508544922
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'val_loss': 1.3518352508544922, 'val_acc': 0.6927138566970825}]